# A Neural Network That Learns How to Add Two Numbers

Okay so... I want to build a neural network that learns to add two numbers. And yeah, I already know computers can add - that's not the point. The point is I want to actually *see* what happens inside a neural network while it's learning. All those words people throw around - neurons, weights, biases, forward pass, backprop - I want to watch them work on something so simple I can't get confused about the problem itself.

If I give the network 2 and 3, I already know the answer is 5. So I'm not going to be sitting there wondering "wait is the model wrong or is the data weird?" - none of that. I can just stare at the weights and the loss and try to build some intuition about what's actually going on.

Some things I'm curious about:
- What does a neuron actually do with the numbers I give it?
- How does the network make a guess?
- Why are the first few guesses so bad?
- How does it know it messed up?
- What actually changes in the weights after each step?
- How does it get better over time?

I'm treating this whole notebook like an experiment. Build it piece by piece, inspect everything at every stage, and hopefully come out the other side with some real intuition.

Oh and - no PyTorch, no TensorFlow. Just Python and NumPy. I want to write the forward pass myself, write the backward pass myself, and actually understand what each line is doing. Also I'm doing this in OOP style (classes and objects) because that's what Karpathy used in his micrograd series and honestly most real codebases use it, so might as well get comfortable with it now.

In [121]:
# Importing the libraries that we need 
import numpy as np

# A Neuron

Before jumping into the whole network, let me start with the smallest piece - one neuron.

A neuron is basically just a tiny math function. It takes some inputs, each input has a weight (how much the neuron "cares" about that input), it multiplies them together, sums everything up, and adds a bias at the end. That's... kind of it. The whole thing is just `(input × weight) + bias` repeated for each input and summed together.

In [122]:
# Let's create Neuron class
class Neuron:
    def __init__(self,weight,bias):
        self.weight=weight
        self.bias=bias
    
    def forward(self,x):
        return x*self.weight+self.bias

# Create a Neuron object and see whether it works
Neuron1=Neuron(2,4) # A neuron with weight 2 and bias 4
result_neuron1=Neuron1.forward(4) #4 is our input in this case and we want to forward it through the neuron
print(result_neuron1)
print(f"This was obtained by: 4*weight={Neuron1.weight} + bias={Neuron1.bias} = {result_neuron1}")

12
This was obtained by: 4*weight=2 + bias=4 = 12


# Our neuron works... but there are two problems

1. When someone wants to add two numbers, they shouldn't have to think about weights and biases at all. Like... the whole point is that the network handles that internally. I should be able to just say "here are two numbers" and the neuron figures out the weights on its own.

2. Right now our `forward` method only takes one value `x`. But addition needs two numbers. So I need to pass in an array of two values instead.

## Fixing these

For problem 1 - I'll use `np.random.randn` to give the neuron random starting weights (one per input, so two weights for our case since we're adding two numbers) and set the bias to 0. The `* np.sqrt(2 / input_size)` part is just to keep the numbers from getting too big or too small. It's a stability thing I picked up from reading around.

For problem 2 - instead of passing a single number `x`, I'll pass an array like `[2, 4]` and use `np.dot` to do the multiplication and summing in one go.

In [123]:
class Neuron:
    def __init__(self, input_size):
        # Use 1-D weights so np.dot returns a scalar, not shape (1,)
        self.weights = np.random.randn(input_size) * np.sqrt(2 / input_size)
        self.bias = 0

    def forward(self, x):
        return np.dot(x, self.weights) + self.bias

In [124]:
Neuron2=Neuron(2)
Neuron2.forward([2,4])

np.float64(1.97949040855703)

# Building a Layer

A layer is just a bunch of neurons grouped together. Why have more than one? Because a single neuron can only pick up on one pattern. Having multiple neurons - each with different random weights - means the layer can spot different things in the same input. More perspectives, basically.

(Also - objects can contain other objects. A Layer holds Neurons. A Network will hold Layers. This nesting thing is kind of the whole idea behind OOP for neural nets.)

In [125]:
class Layer:
    def __init__(self,n_input_size,n_neurons):
        self.neurons=[Neuron(n_input_size) for _ in range(n_neurons)]
    
    def forward(self,x):
        self.last_input=x
        self.z=np.array([neuron.forward(x) for neuron in self.neurons])
        return self.z
    
Layer1=Layer(2,6)
result_Layer1=Layer1.forward([2,4])
print(result_Layer1)

[ 0.96928677  0.37059385 -2.15925809  1.45019719 -3.77849206 -3.44545484]


# Adding a Network class
Now time to build our network class. A network is a group of layers

In [126]:
class Network:
    def __init__(self,layers):
        self.layers=layers
    def forward(self,x):
        for layer in self.layers:
            x=layer.forward(x)
        return x

# NOW......

So we have neurons, layers, and a network. Data can flow through - get multiplied by weights, summed up in each neuron, passed from one layer to the next. But the network isn't *learning* anything yet. The weights are random and they stay random no matter how many times we push data through.

For the network to actually learn, I need four more things:
- **Activation functions** - so the network can learn non-linear stuff (without them, stacking layers is pointless, it all collapses to one big linear equation)
- **A loss function** - something that tells us "how wrong was that guess?"
- **Backpropagation** - figuring out which weights contributed how much to the error
- **Gradient descent** - actually nudging the weights in the right direction

Let me go through each one.

# Activation Functions

So here's something that took me a minute to understand...

Without an activation function, everything a neuron does is linear - multiply, sum, add bias. If you stack a bunch of linear layers on top of each other... you still just get one big linear function. The depth adds nothing. You could replace the whole network with a single layer and get the same result.

Activation functions break this. They introduce non-linearity - they let the network learn curves, boundaries, and patterns that aren't just straight lines.

The one I'm using here is **ReLU** (Rectified Linear Unit). It's dead simple:

```
ReLU(x) = max(0, x)
```

If the input is positive, it passes through unchanged. If it's negative, it becomes zero. That's it. But that tiny "zero out the negatives" behavior is enough to make the network non-linear and actually capable of learning meaningful stuff.

(One thing to note: we only apply ReLU to hidden layers. The output layer stays linear because we want raw numbers out, not numbers that got zeroed by ReLU.)

In [127]:
def relu(x):
    return np.maximum(0,x)

def relu_derivative(x):
    return (x > 0).astype(float)


# Remember that the output layers dont need the activation function, only the hidden layers do. So we will apply the relu function only to the hidden layers and not to the output layer.
# We will find a way of ensuring this later on, but for now we will just apply the relu function to all the layers and then we will find a way of ensuring that it is only applied to the hidden layers.

# Loss Function

The loss function is how the network knows it messed up. It's just the difference between what the network predicted and what the real answer should be.

I'm using **MSE (Mean Squared Error)**:

```
loss = (predicted - actual)²
```

Squaring it does two things: it makes sure the error is always positive (so errors in opposite directions don't cancel out), and it punishes big errors way more than small ones.

The derivative (which we need for backprop) is just `2 * (predicted - actual)`. That derivative tells us which direction to nudge the prediction - and eventually the weights - to make the loss smaller next time.

In [128]:
def mse(predicted_value,true_value):
    return np.mean((predicted_value-true_value)**2)

def mse_derivative(predicted_value, true_value):
    return 2*(predicted_value-true_value)

# Backpropagation and Gradient Descent

Once we know the loss (how wrong we are), we need to figure out *which weights are responsible* and *how much each one should change*. That's backpropagation.

The idea: the error starts at the output layer and flows backward through the network. At each layer, we figure out how much each neuron contributed to the error, and then how much each weight inside that neuron contributed. It's kind of like tracing blame backward through the whole system.

Once we have those gradients (the "blame" numbers for each weight), we use **gradient descent** to actually update:

```
new_weight = old_weight - (learning_rate × gradient)
```

The learning rate controls how big each step is. Too big and we overshoot, too small and it takes forever. I'm using 0.01 which seemed to work after some trial and error.

### Extending our classes to handle activation, backprop, and weight updates

Now I need to go back and add `backward()` methods to Layer and Network. The forward pass was the easy part - data goes in, flows through, prediction comes out. The backward pass is where the actual learning happens, and it's a bit more involved.

In [129]:
class Layer:
    def __init__(self, n_input_size, n_neurons, activation='relu'):
        self.activation = activation
        self.neurons = [Neuron(n_input_size) for _ in range(n_neurons)]

    def forward(self, x):
        self.last_input = x
        self.z = np.array([neuron.forward(x) for neuron in self.neurons])
        # Apply activation only when set to 'relu'; output layers use 'linear'
        if self.activation == 'relu':
            self.a = relu(self.z)
        else:
            self.a = self.z
        return self.a

    def backward(self, grad_output, lr):
        # Gradient through activation
        if self.activation == 'relu':
            dz = grad_output * relu_derivative(self.z)
        else:
            dz = grad_output   # linear: derivative is 1

        # Compute gradient w.r.t. input BEFORE updating weights
        # (W^T @ dz, using the original weights from the forward pass)
        grad_input = np.array([
            sum(n.weights[j] * dz[i] for i, n in enumerate(self.neurons))
            for j in range(len(self.last_input))
        ])

        # Now update weights and biases
        for i, neuron in enumerate(self.neurons):
            neuron.weights -= lr * dz[i] * self.last_input
            neuron.bias    -= lr * dz[i]

        return grad_input

In [130]:
class Network:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, grad, lr):
        for layer in reversed(self.layers):
            grad = layer.backward(grad, lr)

In [131]:
# Generate a proper addition dataset covering a wide range
np.random.seed(42)

# 200 random pairs in range [0, 100]
X_raw = np.random.randint(0, 101, size=(200, 2)).astype(float)
Y_raw = X_raw[:, 0] + X_raw[:, 1]

# Normalize so the network works with small values (roughly 0–1 range)
X_max = X_raw.max()
Y_max = Y_raw.max()
X = X_raw / X_max
Y = Y_raw / Y_max

# Hidden layer uses ReLU; output layer is linear (no activation)
net = Network([Layer(2, 8, activation='relu'), Layer(8, 1, activation='linear')])

lr = 0.01

for epoch in range(2000):
    loss = 0

    for x, y in zip(X, Y):
        # forward
        pred = net.forward(x)[0]

        # loss (on normalized values)
        loss += (pred - y) ** 2

        # gradient of loss wrt output
        grad = 2 * (pred - y)

        # backward
        net.backward(np.array([grad]), lr)

    if epoch % 200 == 0:
        print(f"epoch {epoch:4d}  loss {loss / len(X):.6f}")

print()
print("=== Spot checks on the training data ===")
for i in range(5):
    x_raw, y_raw = X_raw[i], Y_raw[i]
    pred_norm = net.forward(x_raw / X_max)[0]   # normalize input, get normalized output
    pred = pred_norm * Y_max                     # denormalize back to real sum
    print(f"{x_raw[0]:3.0f} + {x_raw[1]:3.0f} = {pred:7.2f}  (true: {y_raw:.0f}, error: {abs(pred - y_raw):.4f})")

epoch    0  loss 0.026375
epoch  200  loss 0.000023
epoch  400  loss 0.000005
epoch  600  loss 0.000002
epoch  800  loss 0.000001
epoch 1000  loss 0.000000
epoch 1200  loss 0.000000
epoch 1400  loss 0.000000
epoch 1600  loss 0.000000
epoch 1800  loss 0.000000

=== Spot checks on the training data ===
 51 +  92 =  143.00  (true: 143, error: 0.0024)
 14 +  71 =   85.00  (true: 85, error: 0.0002)
 60 +  20 =   80.00  (true: 80, error: 0.0026)
 82 +  86 =  168.00  (true: 168, error: 0.0034)
 74 +  74 =  148.00  (true: 148, error: 0.0025)


# Try your own numbers

The network is trained. Let's throw some numbers at it and see if it actually learned to add. Change `a` and `b` below to whatever you want - the model should get pretty close.

In [135]:
# Change these to any two numbers you want to test
a = 30
b = 600

# Normalize the input, get normalized prediction, then denormalize
x_norm = np.array([a, b]) / X_max
pred_norm = net.forward(x_norm)[0]
prediction = pred_norm * Y_max
actual = a + b
error = abs(prediction - actual)

print(f"Model prediction : {prediction:.4f}")
print(f"Actual sum      : {actual}")
print(f"Error           : {error:.6f}")

Model prediction : 630.0269
Actual sum      : 630
Error           : 0.026876
